# Engineering re-run with per-run feasibility

Standalone. Re-runs the four constrained engineering problems storing **every** solution
vector, so feasibility can be assessed per run rather than only for the best.

## Why

The original run (`05_Engineering.ipynb`) stored only `best_x` per algorithm. Verification
showed the static squared penalty does not exclude infeasible solutions: on Speed Reducer
Design none of the fifteen algorithms produced a feasible best solution, and on Pressure
Vessel Design only two did. The reported success rates therefore count marginally
infeasible solutions as successes.

## What changes and what does not

**Unchanged:** the penalised objective handed to every optimiser is byte-identical to
Notebook 05 cell 6 — `1e6 * sum(max(0, g)**2)` added to the raw objective. Population 30,
500 iterations, seeds 42-71, same fifteen competitors plus QSO. **Search behaviour is
therefore exactly as before.**

**Added:** all 30 solution vectors are retained per algorithm-problem pair, and each is
re-evaluated against the raw constraints. This yields, for the first time, per-run
feasibility, feasibility rates, and success rates conditioned on feasibility.

Rolling Element Bearing remains excluded (formulation defect, Section 5.6).

**Runtime** roughly 20-30 minutes. Checkpoints are written per problem, so a disconnect
costs at most one problem.

In [ ]:
# === 1. Setup ===
from google.colab import drive
drive.mount('/content/drive')

import os, json, time
import numpy as np
import pandas as pd

BASE      = '/content/drive/MyDrive/QSO_Research'
SAVE_PATH = f'{BASE}/results/raw/engineering_feasible'
os.makedirs(SAVE_PATH, exist_ok=True)

SEEDS     = list(range(42, 72))     # 30 runs, as in the original
POP_SIZE  = 30
MAX_ITER  = 500
FEAS_TOL  = 1e-6                    # max(0, g_i) must not exceed this
SUCCESS_TOL = 0.01                  # within 1% of the cited optimum

print('output:', SAVE_PATH)

In [ ]:
# === 2. Competitor algorithms (verbatim from 02_Competitors.ipynb) ===
def initialise_population(pop_size, dim, lb, ub, seed=42):
    """Standardised population initialisation for all algorithms."""
    np.random.seed(seed)
    lb = np.full(dim, lb) if np.isscalar(lb) else np.array(lb)
    ub = np.full(dim, ub) if np.isscalar(ub) else np.array(ub)
    X  = np.random.uniform(lb, ub, (pop_size, dim))
    return X, lb, ub


def evaluate_population(func, X):
    """Evaluate fitness for all agents."""
    return np.array([func(X[i]) for i in range(len(X))])


def bound_check(X, lb, ub):
    """Reflect positions back into bounds."""
    return np.clip(X, lb, ub)


def get_best(fitness, X):
    """Return best fitness and position."""
    idx = np.argmin(fitness)
    return fitness[idx], X[idx].copy()


# Standard return format for ALL algorithms:
# (best_fitness, best_position, convergence_curve)
# This uniform interface is critical for fair comparison

def pso(func, lb, ub, dim,
        pop_size=30, max_iter=500,
        w=0.7, c1=1.5, c2=1.5,
        seed=42):
    """
    Particle Swarm Optimisation (Kennedy & Eberhart, 1995)

    Parameters:
    -----------
    w  : float — inertia weight (default 0.7)
    c1 : float — cognitive coefficient (default 1.5)
    c2 : float — social coefficient (default 1.5)
    """
    X, lb, ub = initialise_population(pop_size, dim, lb, ub, seed)

    # Velocities
    V       = np.zeros((pop_size, dim))
    v_max   = 0.2 * (ub - lb)

    # Personal and global bests
    pbest_X = X.copy()
    fitness = evaluate_population(func, X)
    pbest_f = fitness.copy()

    gbest_f, gbest_X = get_best(fitness, X)
    convergence = [gbest_f]

    for t in range(max_iter):
        r1 = np.random.rand(pop_size, dim)
        r2 = np.random.rand(pop_size, dim)

        # Velocity update
        V = (w * V
             + c1 * r1 * (pbest_X - X)
             + c2 * r2 * (gbest_X  - X))
        V = np.clip(V, -v_max, v_max)

        # Position update
        X = X + V
        X = bound_check(X, lb, ub)

        # Fitness evaluation
        fitness = evaluate_population(func, X)

        # Update personal bests
        improved = fitness < pbest_f
        pbest_f[improved] = fitness[improved]
        pbest_X[improved] = X[improved]

        # Update global best
        curr_f, curr_X = get_best(fitness, X)
        if curr_f < gbest_f:
            gbest_f = curr_f
            gbest_X = curr_X.copy()

        convergence.append(gbest_f)

    return gbest_f, gbest_X, convergence


# --- Test ---

def ga(func, lb, ub, dim,
       pop_size=30, max_iter=500,
       cr=0.9, mr=0.01,
       seed=42):
    """
    Genetic Algorithm (Holland, 1992)

    Parameters:
    -----------
    cr : float — crossover rate (default 0.9)
    mr : float — mutation rate (default 0.01)
    """
    X, lb, ub = initialise_population(pop_size, dim, lb, ub, seed)
    fitness    = evaluate_population(func, X)

    gbest_f, gbest_X = get_best(fitness, X)
    convergence = [gbest_f]

    for t in range(max_iter):
        new_X = np.zeros_like(X)

        for i in range(pop_size):
            # ── Tournament selection ──────────────────────────
            t1, t2 = np.random.randint(0, pop_size, 2)
            parent1 = X[t1] if fitness[t1] < fitness[t2] else X[t2]

            t3, t4 = np.random.randint(0, pop_size, 2)
            parent2 = X[t3] if fitness[t3] < fitness[t4] else X[t4]

            # ── Single-point crossover ────────────────────────
            if np.random.rand() < cr:
                point   = np.random.randint(1, dim)
                child   = np.concatenate([
                              parent1[:point],
                              parent2[point:]])
            else:
                child = parent1.copy()

            # ── Gaussian mutation ─────────────────────────────
            mask          = np.random.rand(dim) < mr
            child[mask]  += np.random.normal(
                                0, 0.1*(ub[mask]-lb[mask]))
            child         = np.clip(child, lb, ub)
            new_X[i]      = child

        X       = new_X
        fitness = evaluate_population(func, X)

        # Elite preservation — keep best from previous generation
        worst_idx = np.argmax(fitness)
        if fitness[worst_idx] > gbest_f:
            X[worst_idx]       = gbest_X.copy()
            fitness[worst_idx] = gbest_f

        curr_f, curr_X = get_best(fitness, X)
        if curr_f < gbest_f:
            gbest_f = curr_f
            gbest_X = curr_X.copy()

        convergence.append(gbest_f)

    return gbest_f, gbest_X, convergence


# --- Test ---

def de(func, lb, ub, dim,
       pop_size=30, max_iter=500,
       F=0.5, cr=0.9,
       seed=42):
    """
    Differential Evolution (Storn & Price, 1997)
    DE/rand/1/bin variant.

    Parameters:
    -----------
    F  : float — scaling factor (default 0.5)
             Lower values (0.4-0.6) work better on
             continuous unimodal problems. Original
             paper recommends F in [0.4, 1.0].
    cr : float — crossover rate (default 0.9)

    Note on parameters:
    -------------------
    F=0.5 chosen based on parameter sensitivity analysis
    showing F=0.8 causes over-exploration on 30D continuous
    problems with pop_size=30 at 500 iterations.
    This is consistent with Storn & Price (1997) who note
    F in [0.4, 0.6] works well for most continuous problems.
    """
    X, lb, ub = initialise_population(pop_size, dim, lb, ub, seed)
    fitness    = evaluate_population(func, X)

    gbest_f, gbest_X = get_best(fitness, X)
    convergence = [gbest_f]

    for t in range(max_iter):
        for i in range(pop_size):
            # ── Mutation — DE/rand/1 ──────────────────────────
            idxs = list(range(pop_size))
            idxs.remove(i)
            a, b, c_idx = np.random.choice(idxs, 3, replace=False)

            mutant = X[a] + F * (X[b] - X[c_idx])
            mutant = np.clip(mutant, lb, ub)

            # ── Binomial crossover ────────────────────────────
            cross_mask = np.random.rand(dim) < cr
            # Guarantee at least one dimension crosses over
            cross_mask[np.random.randint(dim)] = True
            trial = np.where(cross_mask, mutant, X[i])

            # ── Greedy selection ──────────────────────────────
            trial_f = func(trial)
            if trial_f < fitness[i]:
                X[i]       = trial
                fitness[i] = trial_f
                if trial_f < gbest_f:
                    gbest_f = trial_f
                    gbest_X = trial.copy()

        convergence.append(gbest_f)

    return gbest_f, gbest_X, convergence


# --- Test ---

def gwo(func, lb, ub, dim,
        pop_size=30, max_iter=500,
        seed=42):
    """
    Grey Wolf Optimiser (Mirjalili et al., 2014)
    """
    X, lb, ub = initialise_population(pop_size, dim, lb, ub, seed)
    fitness    = evaluate_population(func, X)

    # Alpha, beta, delta wolves
    sorted_idx = np.argsort(fitness)
    alpha_f, alpha_X = fitness[sorted_idx[0]], X[sorted_idx[0]].copy()
    beta_f,  beta_X  = fitness[sorted_idx[1]], X[sorted_idx[1]].copy()
    delta_f, delta_X = fitness[sorted_idx[2]], X[sorted_idx[2]].copy()

    convergence = [alpha_f]

    for t in range(max_iter):
        # Linearly decreasing a from 2 to 0
        a = 2 - 2 * (t / max_iter)

        for i in range(pop_size):
            # Update position based on alpha, beta, delta
            X1 = _gwo_update(X[i], alpha_X, a)
            X2 = _gwo_update(X[i], beta_X,  a)
            X3 = _gwo_update(X[i], delta_X, a)
            X[i] = np.clip((X1 + X2 + X3) / 3, lb, ub)

        fitness = evaluate_population(func, X)

        # Update hierarchy
        sorted_idx = np.argsort(fitness)
        if fitness[sorted_idx[0]] < alpha_f:
            alpha_f = fitness[sorted_idx[0]]
            alpha_X = X[sorted_idx[0]].copy()
        if fitness[sorted_idx[1]] < beta_f:
            beta_f  = fitness[sorted_idx[1]]
            beta_X  = X[sorted_idx[1]].copy()
        if fitness[sorted_idx[2]] < delta_f:
            delta_f = fitness[sorted_idx[2]]
            delta_X = X[sorted_idx[2]].copy()

        convergence.append(alpha_f)

    return alpha_f, alpha_X, convergence


def _gwo_update(x, leader, a):
    """Helper — update position toward a leader wolf."""
    r1, r2 = np.random.rand(len(x)), np.random.rand(len(x))
    A = 2 * a * r1 - a
    C = 2 * r2
    D = np.abs(C * leader - x)
    return leader - A * D


# --- Test ---

def woa(func, lb, ub, dim,
        pop_size=30, max_iter=500,
        seed=42):
    """
    Whale Optimisation Algorithm (Mirjalili & Lewis, 2016)
    """
    X, lb, ub = initialise_population(pop_size, dim, lb, ub, seed)
    fitness    = evaluate_population(func, X)

    gbest_f, gbest_X = get_best(fitness, X)
    convergence = [gbest_f]

    for t in range(max_iter):
        a  = 2 - 2 * (t / max_iter)  # Decreases from 2 to 0
        a2 = -1 - (t / max_iter)     # Decreases from -1 to -2

        for i in range(pop_size):
            r  = np.random.rand()
            A  = 2 * a * np.random.rand(dim) - a
            C  = 2 * np.random.rand(dim)
            b  = 1.0   # Spiral shape constant
            l  = (a2 - 1) * np.random.rand() + 1
            p  = np.random.rand()

            if p < 0.5:
                if np.linalg.norm(A) < 1:
                    # Shrinking encircling
                    D       = np.abs(C * gbest_X - X[i])
                    X[i]    = gbest_X - A * D
                else:
                    # Random search
                    rand_X  = X[np.random.randint(pop_size)]
                    D       = np.abs(C * rand_X - X[i])
                    X[i]    = rand_X - A * D
            else:
                # Spiral bubble-net attack
                D       = np.abs(gbest_X - X[i])
                X[i]    = (D * np.exp(b * l)
                           * np.cos(2 * np.pi * l)
                           + gbest_X)

            X[i] = np.clip(X[i], lb, ub)

        fitness = evaluate_population(func, X)

        curr_f, curr_X = get_best(fitness, X)
        if curr_f < gbest_f:
            gbest_f = curr_f
            gbest_X = curr_X.copy()

        convergence.append(gbest_f)

    return gbest_f, gbest_X, convergence


# --- Test ---

def sca(func, lb, ub, dim,
        pop_size=30, max_iter=500,
        seed=42):
    """
    Sine Cosine Algorithm (Mirjalili, 2016)
    """
    X, lb, ub = initialise_population(pop_size, dim, lb, ub, seed)
    fitness    = evaluate_population(func, X)

    gbest_f, gbest_X = get_best(fitness, X)
    convergence = [gbest_f]

    for t in range(max_iter):
        # Decreasing r1 from 2 to 0
        r1 = 2 - 2 * (t / max_iter)

        for i in range(pop_size):
            r2 = 2 * np.pi * np.random.rand(dim)
            r3 = np.random.rand(dim)
            r4 = np.random.rand()

            if r4 < 0.5:
                X[i] = (X[i]
                        + r1 * np.sin(r2)
                        * np.abs(r3 * gbest_X - X[i]))
            else:
                X[i] = (X[i]
                        + r1 * np.cos(r2)
                        * np.abs(r3 * gbest_X - X[i]))

            X[i] = np.clip(X[i], lb, ub)

        fitness = evaluate_population(func, X)

        curr_f, curr_X = get_best(fitness, X)
        if curr_f < gbest_f:
            gbest_f = curr_f
            gbest_X = curr_X.copy()

        convergence.append(gbest_f)

    return gbest_f, gbest_X, convergence


# --- Test ---

def hho(func, lb, ub, dim,
        pop_size=30, max_iter=500,
        seed=42):
    """
    Harris Hawks Optimisation (Heidari et al., 2019)
    """
    X, lb, ub = initialise_population(pop_size, dim, lb, ub, seed)
    fitness    = evaluate_population(func, X)

    gbest_f, gbest_X = get_best(fitness, X)
    convergence = [gbest_f]

    for t in range(max_iter):
        E0 = 2 * np.random.rand() - 1   # Initial energy
        E  = 2 * E0 * (1 - t / max_iter) # Escaping energy

        for i in range(pop_size):
            r = np.random.rand()

            if np.abs(E) >= 1:
                # ── Exploration ───────────────────────────────
                if r >= 0.5:
                    rand_X   = X[np.random.randint(pop_size)]
                    X[i]     = (rand_X
                                - np.random.rand()
                                * np.abs(rand_X
                                - 2 * np.random.rand() * X[i]))
                else:
                    X[i]     = ((gbest_X - np.mean(X, axis=0))
                                - np.random.rand()
                                * (lb + np.random.rand() * (ub - lb)))
            else:
                # ── Exploitation ──────────────────────────────
                J        = 2 * (1 - np.random.rand())
                delta_X  = gbest_X - X[i]

                if r >= 0.5 and np.abs(E) >= 0.5:
                    # Soft besiege
                    X[i] = delta_X - E * np.abs(J * gbest_X - X[i])

                elif r >= 0.5 and np.abs(E) < 0.5:
                    # Hard besiege
                    X[i] = gbest_X - E * np.abs(delta_X)

                elif r < 0.5 and np.abs(E) >= 0.5:
                    # Soft besiege with progressive rapid dives
                    Y = gbest_X - E * np.abs(J * gbest_X - X[i])
                    Z = Y + np.random.rand(dim) * _levy_hho(dim)
                    X[i] = (Y if func(Y) < func(Z) else Z)

                else:
                    # Hard besiege with progressive rapid dives
                    Y = gbest_X - E * np.abs(J * gbest_X - np.mean(X, axis=0))
                    Z = Y + np.random.rand(dim) * _levy_hho(dim)
                    X[i] = (Y if func(Y) < func(Z) else Z)

            X[i] = np.clip(X[i], lb, ub)

        fitness = evaluate_population(func, X)

        curr_f, curr_X = get_best(fitness, X)
        if curr_f < gbest_f:
            gbest_f = curr_f
            gbest_X = curr_X.copy()

        convergence.append(gbest_f)

    return gbest_f, gbest_X, convergence


def _levy_hho(dim, beta=1.5):
    """Lévy flight helper for HHO."""
    from scipy.special import gamma
    sigma = (gamma(1+beta) * np.sin(np.pi*beta/2) /
             (gamma((1+beta)/2) * beta * 2**((beta-1)/2)))**(1/beta)
    u = np.random.normal(0, sigma, dim)
    v = np.random.normal(0, 1, dim)
    return u / (np.abs(v)**(1/beta))


# --- Test ---

def mpa(func, lb, ub, dim,
        pop_size=30, max_iter=500,
        seed=42):
    """
    Marine Predators Algorithm (Faramarzi et al., 2020)
    """
    X, lb, ub = initialise_population(pop_size, dim, lb, ub, seed)
    fitness    = evaluate_population(func, X)

    gbest_f, gbest_X = get_best(fitness, X)

    # Elite matrix — top predator
    Elite   = np.tile(gbest_X, (pop_size, 1))
    convergence = [gbest_f]
    P       = 0.5
    FADs    = 0.2

    for t in range(max_iter):
        CF = (1 - t/max_iter) ** (2*t/max_iter)

        RL = 0.05 * _levy_mpa(pop_size, dim)
        RB = np.random.randn(pop_size, dim)

        for i in range(pop_size):
            r  = np.random.rand()
            R  = np.random.rand(dim)

            if t < max_iter / 3:
                # Phase 1 — High velocity ratio (prey moves faster)
                stepsize   = RB[i] * (Elite[i] - RB[i] * X[i])
                X[i]      += P * stepsize

            elif t < 2 * max_iter / 3:
                if i < pop_size // 2:
                    # Phase 2a — Unit velocity ratio (Lévy)
                    stepsize = RL[i] * (Elite[i] - RL[i] * X[i])
                    X[i]    += P * stepsize
                else:
                    # Phase 2b — Unit velocity ratio (Brownian)
                    stepsize = RB[i] * (RB[i] * Elite[i] - X[i])
                    X[i]    += P * CF * stepsize
            else:
                # Phase 3 — Low velocity ratio (predator moves faster)
                stepsize   = RL[i] * (RL[i] * Elite[i] - X[i])
                X[i]      += P * CF * stepsize

            # FADs effect
            if np.random.rand() < FADs:
                U    = np.random.rand(dim) < FADs
                X[i]+= CF * (lb + np.random.rand(dim)*(ub-lb)) * U

            X[i] = np.clip(X[i], lb, ub)

        fitness = evaluate_population(func, X)

        curr_f, curr_X = get_best(fitness, X)
        if curr_f < gbest_f:
            gbest_f = curr_f
            gbest_X = curr_X.copy()

        # Update elite matrix
        Elite = np.tile(gbest_X, (pop_size, 1))
        convergence.append(gbest_f)

    return gbest_f, gbest_X, convergence


def _levy_mpa(n, d, beta=1.5):
    """Lévy flight helper for MPA."""
    from scipy.special import gamma
    sigma = (gamma(1+beta) * np.sin(np.pi*beta/2) /
             (gamma((1+beta)/2) * beta * 2**((beta-1)/2)))**(1/beta)
    u = np.random.normal(0, sigma, (n, d))
    v = np.random.normal(0, 1, (n, d))
    return u / (np.abs(v)**(1/beta))


# --- Test ---

def bfo(func, lb, ub, dim,
        pop_size=30, max_iter=500,
        n_swim=4, n_tumble=4,
        seed=42):
    """
    Bacterial Foraging Optimisation (Passino, 2002)

    Parameters:
    -----------
    n_swim   : int — swim steps per chemotaxis (default 4)
    n_tumble : int — tumble steps (default 4)
    """
    X, lb, ub = initialise_population(pop_size, dim, lb, ub, seed)
    fitness    = evaluate_population(func, X)

    gbest_f, gbest_X = get_best(fitness, X)
    convergence = [gbest_f]

    step_size = 0.1 * (ub - lb)
    iters_per_cycle = max(1, max_iter // (n_tumble * n_swim + 1))

    for t in range(max_iter):
        for i in range(pop_size):
            # ── Tumble — random direction ─────────────────────
            delta = np.random.randn(dim)
            delta /= (np.linalg.norm(delta) + 1e-10)

            # ── Swim — move in tumble direction ───────────────
            for s in range(n_swim):
                X_new    = X[i] + step_size * delta
                X_new    = np.clip(X_new, lb, ub)
                f_new    = func(X_new)

                if f_new < fitness[i]:
                    X[i]       = X_new
                    fitness[i] = f_new
                    if f_new < gbest_f:
                        gbest_f = f_new
                        gbest_X = X_new.copy()
                else:
                    break

        # ── Reproduction — top half survives ─────────────────
        if t % iters_per_cycle == 0:
            sorted_idx   = np.argsort(fitness)
            X            = np.vstack([
                               X[sorted_idx[:pop_size//2]],
                               X[sorted_idx[:pop_size//2]]
                           ])
            fitness      = np.concatenate([
                               fitness[sorted_idx[:pop_size//2]],
                               fitness[sorted_idx[:pop_size//2]]
                           ])

        # Decrease step size over time
        step_size *= 0.99

        convergence.append(gbest_f)

    return gbest_f, gbest_X, convergence


# --- Test ---

def qbso(func, lb, ub, dim,
         pop_size=30, max_iter=500,
         qs_threshold=0.5,
         seed=42):
    """
    Quorum Sensing Bacterial Swarm Optimisation (QBSO)
    Based on: Li et al. (2019)

    QS used as enhancement to bacterial swarm —
    NOT as standalone framework (key distinction from QSO)
    """
    X, lb, ub = initialise_population(pop_size, dim, lb, ub, seed)
    fitness    = evaluate_population(func, X)

    gbest_f, gbest_X = get_best(fitness, X)
    convergence = [gbest_f]

    step_size = 0.1 * (ub - lb)

    for t in range(max_iter):
        # ── Compute quorum signal ─────────────────────────────
        f_worst = np.max(fitness)
        f_best  = np.min(fitness)
        epsilon = 1e-10

        if f_worst - f_best < epsilon:
            qs_signal = 0.5
        else:
            qs_signal = np.mean(
                (f_worst - fitness) / (f_worst - f_best + epsilon))

        for i in range(pop_size):
            delta = np.random.randn(dim)
            delta /= (np.linalg.norm(delta) + 1e-10)

            if qs_signal >= qs_threshold:
                # QS triggered — move toward global best
                direction = gbest_X - X[i]
                norm      = np.linalg.norm(direction) + 1e-10
                X[i]     += step_size * (direction/norm)
            else:
                # QS not triggered — random walk
                X[i]     += step_size * delta

            X[i] = np.clip(X[i], lb, ub)

        fitness = evaluate_population(func, X)

        curr_f, curr_X = get_best(fitness, X)
        if curr_f < gbest_f:
            gbest_f = curr_f
            gbest_X = curr_X.copy()

        step_size *= 0.995
        convergence.append(gbest_f)

    return gbest_f, gbest_X, convergence


def qbho(func, lb, ub, dim,
         pop_size=30, max_iter=500,
         qs_threshold=0.5,
         seed=42):
    """
    Quorum Sensing Bacterial Horde Optimisation (QBHO)
    Based on: Alzaqebah et al. (2023)

    QS used to identify optimal bacterial positions —
    NOT as standalone framework (key distinction from QSO)
    """
    X, lb, ub = initialise_population(pop_size, dim, lb, ub, seed)
    fitness    = evaluate_population(func, X)

    gbest_f, gbest_X = get_best(fitness, X)
    convergence = [gbest_f]

    for t in range(max_iter):
        # ── Quorum detection ──────────────────────────────────
        f_worst   = np.max(fitness)
        f_best    = np.min(fitness)
        epsilon   = 1e-10

        qs_signal = np.mean(
            (f_worst - fitness) / (f_worst - f_best + epsilon + 1e-10))

        # ── Worst position used as reference (per QBHO paper) ─
        worst_idx = np.argmax(fitness)

        for i in range(pop_size):
            r1 = np.random.rand(dim)
            r2 = np.random.rand(dim)

            if qs_signal >= qs_threshold:
                # Quorum active — avoid worst, move to best
                X[i] = (X[i]
                        + r1 * (gbest_X - X[i])
                        - r2 * (X[worst_idx] - X[i]))
            else:
                # Quorum inactive — standard foraging
                rand_X = X[np.random.randint(pop_size)]
                X[i]   = X[i] + r1 * (rand_X - X[i])

            X[i] = np.clip(X[i], lb, ub)

        fitness = evaluate_population(func, X)

        curr_f, curr_X = get_best(fitness, X)
        if curr_f < gbest_f:
            gbest_f = curr_f
            gbest_X = curr_X.copy()

        convergence.append(gbest_f)

    return gbest_f, gbest_X, convergence


# --- Tests ---

def dbo(func, lb, ub, dim,
        pop_size=30, max_iter=500,
        seed=42):
    """
    Dung Beetle Optimisation (Xue & Shen, 2022)

    Four beetle roles:
    - Ball-rollers  : navigate using celestial cues (exploration)
    - Dancers       : reorient when lost (escape local optima)
    - Foragers      : search near best site (exploitation)
    - Brood-stealers: compete for best positions (intensification)
    """
    X, lb, ub = initialise_population(pop_size, dim, lb, ub, seed)
    fitness    = evaluate_population(func, X)

    gbest_f, gbest_X = get_best(fitness, X)
    convergence = [gbest_f]

    # Population split into 4 roles
    n_rollers  = pop_size // 4
    n_dancers  = pop_size // 4
    n_foragers = pop_size // 4
    n_thieves  = pop_size - n_rollers - n_dancers - n_foragers

    # Role index boundaries
    r_end = n_rollers
    d_end = n_rollers + n_dancers
    f_end = n_rollers + n_dancers + n_foragers

    for t in range(max_iter):
        R  = 1 - t / max_iter       # Decreasing radius
        CF = (1 - t/max_iter) ** 2  # Convergence factor

        # ── Ball-rolling beetles (exploration) ────────────────────
        for i in range(r_end):
            if np.random.rand() > 0.9:
                # Dancing reorientation
                X[i] = X[i] + np.tan(
                    np.random.rand(dim)) * np.abs(X[i] - gbest_X)
            else:
                # Navigate toward best with decreasing radius
                r1   = np.random.rand(dim)
                X[i] = X[i] + R * r1 * (gbest_X - X[i])
            X[i] = np.clip(X[i], lb, ub)

        # ── Dancing beetles (escape local optima) ─────────────────
        for i in range(r_end, d_end):
            r1   = np.random.rand(dim)
            X[i] = gbest_X + r1 * np.abs(X[i] - gbest_X) * CF
            X[i] = np.clip(X[i], lb, ub)

        # ── Foraging beetles (exploitation) — FIXED ───────────────
        for i in range(d_end, f_end):
            r1   = np.random.rand(dim)
            r2   = np.random.rand(dim)
            # Move toward global best with random perturbation
            X[i] = (X[i]
                    + r1 * (gbest_X - X[i])
                    + r2 * CF * np.random.randn(dim))
            X[i] = np.clip(X[i], lb, ub)

        # ── Brood-stealing beetles (intensification) ──────────────
        for i in range(f_end, pop_size):
            r1   = np.random.rand(dim)
            r2   = np.random.rand(dim)
            # Steal position near global best
            X[i] = (gbest_X
                    + r1 * CF * (X[i] - gbest_X)
                    + r2 * np.random.randn(dim) * R)
            X[i] = np.clip(X[i], lb, ub)

        # ── Evaluate & update best ────────────────────────────────
        fitness = evaluate_population(func, X)

        # Elite preservation
        worst_idx = np.argmax(fitness)
        if fitness[worst_idx] > gbest_f:
            X[worst_idx]       = gbest_X.copy()
            fitness[worst_idx] = gbest_f

        curr_f, curr_X = get_best(fitness, X)
        if curr_f < gbest_f:
            gbest_f = curr_f
            gbest_X = curr_X.copy()

        convergence.append(gbest_f)

    return gbest_f, gbest_X, convergence


# --- Test ---

def poa(func, lb, ub, dim,
        pop_size=30, max_iter=500,
        seed=42):
    """
    Pelican Optimisation Algorithm (Trojovský & Dehghani, 2022)
    """
    X, lb, ub = initialise_population(pop_size, dim, lb, ub, seed)
    fitness    = evaluate_population(func, X)

    gbest_f, gbest_X = get_best(fitness, X)
    convergence = [gbest_f]

    for t in range(max_iter):
        for i in range(pop_size):
            # ── Phase 1: Moving toward prey ───────────────────
            # Random prey selection
            prey_idx  = np.random.randint(pop_size)
            prey_X    = X[prey_idx]
            prey_f    = fitness[prey_idx]

            X1 = X[i] + np.random.rand(dim) * (
                prey_X - np.random.randint(1, 3) * X[i])
            X1 = np.clip(X1, lb, ub)
            f1 = func(X1)

            if f1 < fitness[i]:
                X[i]       = X1
                fitness[i] = f1

            # ── Phase 2: Winging on water surface ─────────────
            R    = 0.2 * (1 - t / max_iter)
            X2   = X[i] + R * (2 * np.random.rand(dim) - 1) * X[i]
            X2   = np.clip(X2, lb, ub)
            f2   = func(X2)

            if f2 < fitness[i]:
                X[i]       = X2
                fitness[i] = f2

            if fitness[i] < gbest_f:
                gbest_f = fitness[i]
                gbest_X = X[i].copy()

        convergence.append(gbest_f)

    return gbest_f, gbest_X, convergence


def evo(func, lb, ub, dim,
        pop_size=30, max_iter=500,
        seed=42):
    """
    Electric Eel Foraging Optimiser (EVO)
    Based on: Wang et al. (2024)
    """
    X, lb, ub = initialise_population(pop_size, dim, lb, ub, seed)
    fitness    = evaluate_population(func, X)

    gbest_f, gbest_X = get_best(fitness, X)
    convergence = [gbest_f]

    for t in range(max_iter):
        a = 2 * (1 - t / max_iter)  # Decreasing factor

        for i in range(pop_size):
            r1 = np.random.rand(dim)
            r2 = np.random.rand(dim)

            # ── Electric discharge hunting ────────────────────
            if np.random.rand() < 0.5:
                # Discharge toward best
                X[i] = (X[i]
                        + a * r1 * (gbest_X - X[i])
                        + (1-a) * r2 * (
                            X[np.random.randint(pop_size)] - X[i]))
            else:
                # Passive drift with random component
                beta   = np.random.randn(dim)
                X[i]   = (gbest_X
                          + beta * np.abs(gbest_X - X[i]) * (1 - a))

            X[i] = np.clip(X[i], lb, ub)

        fitness = evaluate_population(func, X)

        curr_f, curr_X = get_best(fitness, X)
        if curr_f < gbest_f:
            gbest_f = curr_f
            gbest_X = curr_X.copy()

        convergence.append(gbest_f)

    return gbest_f, gbest_X, convergence


# --- Tests ---

def _levy_gjo(n, d, beta=1.5):
    """Lévy flight helper for GJO."""
    from scipy.special import gamma
    sigma = (gamma(1+beta) * np.sin(np.pi*beta/2) /
             (gamma((1+beta)/2) * beta
              * 2**((beta-1)/2)))**(1/beta)
    u = np.random.normal(0, sigma, (n, d))
    v = np.random.normal(0, 1, (n, d))
    return u / (np.abs(v)**(1/beta))


def gjo(func, lb, ub, dim,
        pop_size=30, max_iter=500,
        seed=42):
    """
    Golden Jackal Optimizer (Chopra & Ansari, 2022)
    Published: Expert Systems with Applications, 198, 116924

    Models male and female jackal hunting behaviour:
    - Male jackal: tracks prey (global best)
    - Female jackal: supports male (second best)
    - Prey escape energy decreases over iterations
    """
    X, lb, ub = initialise_population(
                    pop_size, dim, lb, ub, seed)
    fitness    = evaluate_population(func, X)

    # Male and female jackal (best two solutions)
    sorted_idx = np.argsort(fitness)
    male_pos   = X[sorted_idx[0]].copy()
    male_f     = fitness[sorted_idx[0]]
    female_pos = X[sorted_idx[1]].copy()
    female_f   = fitness[sorted_idx[1]]

    gbest_f     = male_f
    gbest_X     = male_pos.copy()
    convergence = [gbest_f]

    for t in range(max_iter):
        E1 = 1.5 * (1 - t / max_iter)
        RL = 0.05 * _levy_gjo(pop_size, dim)

        for i in range(pop_size):
            E0 = 2 * np.random.rand() - 1
            E  = E1 * E0

            # Update toward male jackal
            D_male   = np.abs(RL[i] * male_pos - X[i])
            X1       = male_pos - E * D_male

            # Update toward female jackal
            D_female = np.abs(RL[i] * female_pos - X[i])
            X2       = female_pos - E * D_female

            # Average of both updates
            X[i] = np.clip((X1 + X2) / 2, lb, ub)

        fitness = evaluate_population(func, X)

        # Update male and female jackals
        sorted_idx = np.argsort(fitness)

        if fitness[sorted_idx[0]] < male_f:
            male_f   = fitness[sorted_idx[0]]
            male_pos = X[sorted_idx[0]].copy()

        if fitness[sorted_idx[1]] < female_f:
            female_f   = fitness[sorted_idx[1]]
            female_pos = X[sorted_idx[1]].copy()

        if male_f < gbest_f:
            gbest_f = male_f
            gbest_X = male_pos.copy()

        convergence.append(gbest_f)

    return gbest_f, gbest_X, convergence


# --- Test ---


In [ ]:
# === 3. QSO (verbatim from qso.py) ===

import numpy as np

def levy_flight(n, d, alpha=1.25):
    from scipy.special import gamma
    sigma_u = (
        gamma(1 + alpha) * np.sin(np.pi * alpha / 2) /
        (gamma((1 + alpha) / 2) * alpha * 2**((alpha-1)/2))
    ) ** (1/alpha)
    u = np.random.normal(0, sigma_u, (n, d))
    v = np.random.normal(0, 1.0, (n, d))
    return u / (np.abs(v) ** (1/alpha))

def clip_to_bounds(x, lb, ub):
    repair_lb = lb + np.random.rand(*x.shape) * (ub - lb) * 0.1
    repair_ub = ub - np.random.rand(*x.shape) * (ub - lb) * 0.1
    x = np.where(x < lb, repair_lb, x)
    x = np.where(x > ub, repair_ub, x)
    return x

def compute_ai_concentration(fitness_values, f_best, f_worst):
    epsilon = 1e-10
    if (f_worst - f_best) < epsilon:
        return 0.5
    phi = (f_worst - fitness_values) / (f_worst - f_best + epsilon)
    return float(np.clip(np.mean(phi), 0.0, 1.0))

def adaptive_threshold(t, max_iter, theta_min=0.3, theta_max=0.7):
    return float(theta_max - (theta_max - theta_min)
                 * (t / max_iter))

def exploration_phase(X, lb, ub, alpha=1.25):
    n, d = X.shape
    r1 = np.random.rand(n, d)
    r2 = np.random.rand(n, d)
    X_rand = X[np.random.randint(0, n, size=n)]
    L = levy_flight(n, d, alpha)
    scale = (ub - lb) * 0.01
    L_scaled = np.clip(L * scale, -0.5*(ub-lb), 0.5*(ub-lb))
    X_new = X + r1*(X_rand - X) + r2*L_scaled
    return clip_to_bounds(X_new, lb, ub)

def exploitation_phase(X, X_best, lb, ub):
    n, d = X.shape
    r3 = np.random.rand(n, d)
    r4 = np.random.rand(n, d)
    X_colony = np.mean(X, axis=0)
    X_new = (X
             + r3 * (X_best   - X)
             + r4 * (X_colony - X))
    return clip_to_bounds(X_new, lb, ub)

def apply_ai_decay(C, lambda_=0.05):
    return float(np.clip(C * np.exp(-lambda_), 0.0, 1.0))

def qso(func, lb, ub, dim,
        pop_size=30, max_iter=500,
        theta_min=0.3, theta_max=0.7,
        lambda_=0.05, tau=10, alpha=1.25,
        seed=42, verbose=False):
    np.random.seed(seed)
    lb = np.full(dim, lb) if np.isscalar(lb) else np.array(lb)
    ub = np.full(dim, ub) if np.isscalar(ub) else np.array(ub)
    X = np.random.uniform(lb, ub, (pop_size, dim))
    fitness = np.array([func(X[i]) for i in range(pop_size)])
    best_idx      = np.argmin(fitness)
    best_fitness  = fitness[best_idx]
    best_position = X[best_idx].copy()
    theta_t = adaptive_threshold(
                  0, max_iter, theta_min, theta_max)
    C = compute_ai_concentration(
            fitness, best_fitness, np.max(fitness))
    convergence    = [best_fitness]
    diversity      = [np.mean(np.std(X, axis=0))]
    quorum_history = [C]
    phase_history  = [1 if C >= theta_t else 0]
    theta_history  = [theta_t]
    no_improve_count = 0
    for t in range(max_iter):
        theta_t = adaptive_threshold(
                      t, max_iter, theta_min, theta_max)
        if C >= theta_t:
            X     = exploitation_phase(
                        X, best_position, lb, ub)
            phase = 1
        else:
            X     = exploration_phase(X, lb, ub, alpha)
            phase = 0
        fitness = np.array([func(X[i]) for i in range(pop_size)])
        worst_idx = np.argmax(fitness)
        if fitness[worst_idx] > best_fitness:
            X[worst_idx]       = best_position.copy()
            fitness[worst_idx] = best_fitness
        current_best_idx     = np.argmin(fitness)
        current_best_fitness = fitness[current_best_idx]
        if current_best_fitness < best_fitness:
            best_fitness     = current_best_fitness
            best_position    = X[current_best_idx].copy()
            no_improve_count = 0
        else:
            no_improve_count += 1
        C = compute_ai_concentration(
                fitness, best_fitness, np.max(fitness))
        if no_improve_count >= tau:
            C = apply_ai_decay(C, lambda_)
            no_improve_count = 0
        convergence.append(best_fitness)
        diversity.append(np.mean(np.std(X, axis=0)))
        quorum_history.append(C)
        phase_history.append(phase)
        theta_history.append(theta_t)
        if verbose and (t+1) % 100 == 0:
            print(f"Iter {t+1}/{max_iter} | "
                  f"Best: {best_fitness:.6e} | "
                  f"C: {C:.3f} | theta: {theta_t:.3f}")
    return (best_fitness, best_position, convergence,
            diversity, quorum_history, phase_history,
            theta_history)


In [ ]:
# === 4. Algorithm registry (identical to 05_Engineering.ipynb cell 5) ===
qso_func = qso

ALGORITHM_REGISTRY = {
    'QSO':  lambda f, lb, ub, d, s: qso_func(f, lb, ub, d, pop_size=POP_SIZE, max_iter=MAX_ITER,
                theta_min=0.3, theta_max=0.7, lambda_=0.05, tau=10, alpha=1.25, seed=s),
    'PSO':  lambda f, lb, ub, d, s: pso(f, lb, ub, d, pop_size=POP_SIZE, max_iter=MAX_ITER,
                w=0.7, c1=1.5, c2=1.5, seed=s),
    'GA':   lambda f, lb, ub, d, s: ga(f, lb, ub, d, pop_size=POP_SIZE, max_iter=MAX_ITER,
                cr=0.9, mr=0.01, seed=s),
    'DE':   lambda f, lb, ub, d, s: de(f, lb, ub, d, pop_size=POP_SIZE, max_iter=MAX_ITER,
                F=0.5, cr=0.9, seed=s),
    'GWO':  lambda f, lb, ub, d, s: gwo(f, lb, ub, d, pop_size=POP_SIZE, max_iter=MAX_ITER, seed=s),
    'WOA':  lambda f, lb, ub, d, s: woa(f, lb, ub, d, pop_size=POP_SIZE, max_iter=MAX_ITER, seed=s),
    'SCA':  lambda f, lb, ub, d, s: sca(f, lb, ub, d, pop_size=POP_SIZE, max_iter=MAX_ITER, seed=s),
    'HHO':  lambda f, lb, ub, d, s: hho(f, lb, ub, d, pop_size=POP_SIZE, max_iter=MAX_ITER, seed=s),
    'MPA':  lambda f, lb, ub, d, s: mpa(f, lb, ub, d, pop_size=POP_SIZE, max_iter=MAX_ITER, seed=s),
    'BFO':  lambda f, lb, ub, d, s: bfo(f, lb, ub, d, pop_size=POP_SIZE, max_iter=MAX_ITER, seed=s),
    'QBSO': lambda f, lb, ub, d, s: qbso(f, lb, ub, d, pop_size=POP_SIZE, max_iter=MAX_ITER, seed=s),
    'QBHO': lambda f, lb, ub, d, s: qbho(f, lb, ub, d, pop_size=POP_SIZE, max_iter=MAX_ITER, seed=s),
    'DBO':  lambda f, lb, ub, d, s: dbo(f, lb, ub, d, pop_size=POP_SIZE, max_iter=MAX_ITER, seed=s),
    'POA':  lambda f, lb, ub, d, s: poa(f, lb, ub, d, pop_size=POP_SIZE, max_iter=MAX_ITER, seed=s),
    'EVO':  lambda f, lb, ub, d, s: evo(f, lb, ub, d, pop_size=POP_SIZE, max_iter=MAX_ITER, seed=s),
    'GJO':  lambda f, lb, ub, d, s: gjo(f, lb, ub, d, pop_size=POP_SIZE, max_iter=MAX_ITER, seed=s),
}
print(f'{len(ALGORITHM_REGISTRY)} algorithms registered')

sphere = lambda x: float(np.sum(x**2))
bad = []
for a, fn in ALGORITHM_REGISTRY.items():
    try:
        r = fn(sphere, -100, 100, 10, 42); float(r[0])
    except Exception as e:
        bad.append((a, str(e)[:60]))
print('failures:', bad if bad else 'none')

In [ ]:
# === 5. Engineering problems: raw objective and constraints, separated ===
# Objective and constraint expressions are verbatim from 05_Engineering.ipynb cell 6.
# The penalised wrapper reproduces that cell exactly, so search behaviour is unchanged.

def wbd_parts(x):
    h, l, t, b = x
    P, L, E, G = 6000, 14, 30e6, 12e6
    t_m, s_m, d_m = 13600, 30000, 0.25
    M = P*(L + l/2)
    R = np.sqrt(l**2/4 + ((h+t)/2)**2)
    J = 2*(np.sqrt(2)*h*l*(l**2/12 + ((h+t)/2)**2))
    t1 = P/(np.sqrt(2)*h*l); t2 = M*R/J
    tau = np.sqrt(t1**2 + 2*t1*t2*l/(2*R) + t2**2)
    sigma = 6*P*L/(b*t**2)
    delta = 6*P*L**3/(E*b*t**3)
    Pc = 4.013*E*np.sqrt(t**2*b**6/36)/L**2*(1 - t/(2*L)*np.sqrt(E/(4*G)))
    f = 1.10471*h**2*l + 0.04811*t*b*(14+l)
    g = [tau - t_m, sigma - s_m, h - b,
         0.10471*h**2 + 0.04811*t*b*(14+l) - 5.0,
         0.125 - h, delta - d_m, P - Pc]
    return f, np.array(g)

def pvd_parts(x):
    Ts, Th, R, L = x
    f = 0.6224*Ts*R*L + 1.7781*Th*R**2 + 3.1661*Ts**2*L + 19.84*Ts**2*R
    g = [-Ts + 0.0193*R, -Th + 0.00954*R,
         -np.pi*R**2*L - (4/3)*np.pi*R**3 + 1296000, L - 240]
    return f, np.array(g)

def tcsd_parts(x):
    d, D, N = x
    f = (N + 2)*D*d**2
    g = [1 - D**3*N/(71785*d**4),
         (4*D**2 - d*D)/(12566*(D*d**3 - d**4)) + 1/(5108*d**2) - 1,
         1 - 140.45*d/(D**2*N),
         (D + d)/1.5 - 1]
    return f, np.array(g)

def srd_parts(x):
    b, m, z, l1, l2, d1, d2 = x
    f = (0.7854*b*m**2*(3.3333*z**2 + 14.9334*z - 43.0934)
         - 1.508*b*(d1**2 + d2**2) + 7.4777*(d1**3 + d2**3)
         + 0.7854*(l1*d1**2 + l2*d2**2))
    g = [27/(b*m**2*z) - 1, 397.5/(b*m**2*z**2) - 1,
         1.93*l1**3/(m*z*d1**4) - 1, 1.93*l2**3/(m*z*d2**4) - 1,
         np.sqrt((745*l1/(m*z))**2 + 16.9e6)/(110*d1**3) - 1,
         np.sqrt((745*l2/(m*z))**2 + 157.5e6)/(85*d2**3) - 1,
         m*z/40 - 1, 5*m/b - 1, b/(12*m) - 1,
         1.5*d1/l1 - 1, 1.1*d2/l2 - 1]
    return f, np.array(g)

def penalised(parts_fn):
    """Reproduces cell 6 exactly: f + 1e6 * sum(max(0, g)**2)."""
    def wrapped(x):
        f, g = parts_fn(x)
        return f + 1e6*np.sum(np.maximum(0, g)**2)
    return wrapped

PROBLEMS = {
 'WBD':  dict(parts=wbd_parts,  lb=np.array([0.1,0.1,0.1,0.1]),
              ub=np.array([2.0,10.0,10.0,2.0]), dim=4, optimum=1.7248,
              desc='Welded Beam Design'),
 'PVD':  dict(parts=pvd_parts,  lb=np.array([0.0625,0.0625,10.0,10.0]),
              ub=np.array([6.1875,6.1875,200.0,200.0]), dim=4, optimum=5804.45,
              desc='Pressure Vessel Design'),
 'TCSD': dict(parts=tcsd_parts, lb=np.array([0.05,0.25,2.0]),
              ub=np.array([2.00,1.30,15.0]), dim=3, optimum=0.012665,
              desc='Tension/Compression Spring Design'),
 'SRD':  dict(parts=srd_parts,  lb=np.array([2.6,0.7,17,7.3,7.3,2.9,5.0]),
              ub=np.array([3.6,0.8,28,8.3,8.3,3.9,5.5]), dim=7, optimum=2994.47,
              desc='Speed Reducer Design'),
}
for k, v in PROBLEMS.items():
    v['func'] = penalised(v['parts'])
print('4 problems defined (REB excluded — formulation defect)')

In [ ]:
# === 6. Run: every solution vector retained ===
records = []
t_start = time.time()

for pname, p in PROBLEMS.items():
    chk = f'{SAVE_PATH}/checkpoint_{pname}.json'
    if os.path.exists(chk):
        records += json.load(open(chk))
        print(f'{pname}: loaded from checkpoint')
        continue

    print(f'\n── {pname}: {p["desc"]} ──', flush=True)
    prob_records = []
    for algo, fn in ALGORITHM_REGISTRY.items():
        for seed in SEEDS:
            try:
                res = fn(p['func'], p['lb'], p['ub'], p['dim'], seed)
                x = np.asarray(res[1], dtype=float)
                pen = float(res[0])
                f_raw, g = p['parts'](x)
                viol = np.maximum(0.0, g)
            except Exception as e:
                prob_records.append(dict(problem=pname, algo=algo, seed=seed,
                    penalised=float('inf'), raw=float('inf'), max_violation=float('inf'),
                    feasible=False, x=[], error=str(e)[:80]))
                continue
            prob_records.append(dict(
                problem=pname, algo=algo, seed=seed,
                penalised=pen, raw=float(f_raw),
                max_violation=float(viol.max()),
                n_violated=int((viol > FEAS_TOL).sum()),
                feasible=bool(viol.max() <= FEAS_TOL),
                x=x.tolist()))
        fe = [r for r in prob_records if r['algo']==algo and r['feasible']]
        print(f'  {algo:<6} feasible {len(fe):>2}/30' +
              (f'  best feasible {min(r["raw"] for r in fe):.6g}' if fe else '  — none feasible'),
              flush=True)
    json.dump(prob_records, open(chk, 'w'))
    records += prob_records
    print(f'  checkpoint saved  [{(time.time()-t_start)/60:.1f} min]')

json.dump(records, open(f'{SAVE_PATH}/all_runs.json', 'w'))
print(f'\nDONE — {len(records)} runs in {(time.time()-t_start)/60:.1f} min')

In [ ]:
# === 7. Feasibility-conditioned results (replacement Tables 12 and 13) ===
df = pd.DataFrame([{k: v for k, v in r.items() if k != 'x'} for r in records])

rows = []
for pname, p in PROBLEMS.items():
    sub = df[df.problem == pname]
    for algo in ALGORITHM_REGISTRY:
        a = sub[sub.algo == algo]
        fe = a[a.feasible]
        opt = p['optimum']
        rows.append(dict(
            problem=pname, algo=algo,
            feasibility_rate=len(fe)/len(a) if len(a) else 0.0,
            best_feasible=fe.raw.min() if len(fe) else np.nan,
            mean_feasible=fe.raw.mean() if len(fe) else np.nan,
            std_feasible=fe.raw.std(ddof=1) if len(fe) > 1 else np.nan,
            success_rate=float((fe.raw <= opt*(1+SUCCESS_TOL)).sum())/len(a) if len(a) else 0.0,
            best_penalised_old=a.penalised.min(),
        ))
summary = pd.DataFrame(rows)

# rank on mean of feasible solutions; algorithms with no feasible run rank last
for pname in PROBLEMS:
    m = summary.problem == pname
    summary.loc[m, 'rank'] = summary.loc[m, 'mean_feasible'].rank(method='min', na_option='bottom')

pd.set_option('display.width', 220, 'display.max_rows', 100)
for pname, p in PROBLEMS.items():
    print(f'\n{"="*78}\n{pname} — {p["desc"]}   (cited optimum {p["optimum"]})\n{"="*78}')
    s = summary[summary.problem == pname].sort_values('rank')
    print(s[['algo','feasibility_rate','best_feasible','mean_feasible','std_feasible',
             'success_rate','rank']].to_string(index=False))

summary.to_csv(f'{SAVE_PATH}/summary_feasible.csv', index=False)
print(f'\nsaved: {SAVE_PATH}/summary_feasible.csv')

In [ ]:
# === 8. What changed, and QSO's position ===
print('FEASIBILITY RATE BY PROBLEM (mean across algorithms)')
print(summary.groupby('problem').feasibility_rate.agg(['mean','min','max']).round(3).to_string())
print()
print('QSO RANK ON FEASIBLE SOLUTIONS ONLY')
q = summary[summary.algo == 'QSO'][['problem','feasibility_rate','best_feasible',
                                    'mean_feasible','success_rate','rank']]
print(q.to_string(index=False))
print()
print('Problems where fewer than half of all runs are feasible for any algorithm:')
weak = summary.groupby('problem').feasibility_rate.max()
print([p for p, v in weak.items() if v < 0.5] or '  none')
print()
print('NOTE FOR THE MANUSCRIPT')
print('  success_rate here requires BOTH feasibility at 1e-6 AND being within')
print(f'  {SUCCESS_TOL:.0%} of the cited optimum. The original Table 13 required neither.')

## Using the output

`summary_feasible.csv` replaces Tables 12 and 13. The columns to use:

- `feasibility_rate` — proportion of the 30 runs producing a feasible solution. This is
  new information and belongs in the manuscript in its own right.
- `best_feasible` / `mean_feasible` / `std_feasible` — computed over feasible runs only.
  These replace the penalised values in Table 12.
- `success_rate` — now requires feasibility **and** proximity to the optimum. This
  replaces Table 13.
- `rank` — recomputed on `mean_feasible`; algorithms with no feasible run rank last.

If a problem shows near-zero feasibility across all sixteen algorithms, that is a finding
about the problem formulation or the penalty coefficient, not about the algorithms, and
Section 5.7 should say so.

`all_runs.json` holds every solution vector and should go in the repository cited in
Section 4.5.